<a href="https://colab.research.google.com/github/malcommathela/Data-Analysis/blob/main/Pandas_Filters.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# 🔍 Pandas Filtering: Selecting Data by Conditions

## What You Will Learn
Filtering is the heart of data analysis — isolating the rows that matter. This notebook covers every major filtering technique in pandas:

1. **Boolean indexing** — the foundation of all filtering
2. **Comparison operators** (`>`, `<`, `==`, `!=`)
3. **Multiple conditions** with `&` (AND) and `|` (OR)
4. **`isin()`** — filtering by membership in a list
5. **`between()`** — range-based filtering
6. **String methods** (`str.contains()`, `str.startswith()`)
7. **`query()`** — SQL-like filtering syntax
8. **`filter()`** — selecting by label patterns
9. **`where()`** — conditional replacement
10. **Negation** with `~`

---

## 1. Import Pandas & Load Data

We will reuse the `products.csv` dataset from the previous notebooks.

In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv('products.csv')

print(f"Dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns")
df

## 2. Boolean Indexing — The Core Concept

Every filter in pandas starts with a **boolean mask**: a Series of `True`/`False` values, one per row. When you pass this mask inside `df[mask]`, pandas keeps only the `True` rows.

**Visual intuition:**
```
Row 0: True   → Keep (Wireless Mouse)
Row 1: False  → Drop (Mechanical Keyboard)
Row 2: True   → Keep (USB-C Hub)
...
```

Let's build a mask and inspect it before applying it.

In [ ]:
# Step 1: Create a boolean mask
mask = df['Stock'] > 100

print("Boolean mask (first 10 rows):")
print(mask.head(10))
print(f"\nTrue count: {mask.sum()} rows match the condition")
print(f"False count: {(~mask).sum()} rows do not match")

## 3. Applying a Single Condition

Pass the mask directly into the DataFrame to filter. You can also select specific columns with `.loc[mask, columns]`.

In [ ]:
# Filter: products with stock greater than 100
high_stock = df[df['Stock'] > 100]

print(f"Products with stock > 100: {len(high_stock)}")
display(high_stock[['Name', 'Brand', 'Stock', 'Price']])

## 4. Comparison Operators

Pandas supports all standard Python comparison operators on Series:

| Operator | Meaning | Example |
|----------|---------|---------|
| `>` | Greater than | `df['Price'] > 100` |
| `<` | Less than | `df['Stock'] < 50` |
| `>=` | Greater than or equal | `df['Rating'] >= 4.5` |
| `<=` | Less than or equal | `df['Price'] <= 50` |
| `==` | Equal to | `df['Color'] == 'Black'` |
| `!=` | Not equal to | `df['Brand'] != 'Logitech'` |

In [ ]:
# Example: Premium products (price > $100)
premium = df[df['Price'] > 100]
print(f"Premium products (Price > $100): {len(premium)}")
display(premium[['Name', 'Brand', 'Price', 'Rating']])

# Example: Non-Logitech products
non_logitech = df[df['Brand'] != 'Logitech']
print(f"\nNon-Logitech products: {len(non_logitech)}")
display(non_logitech[['Name', 'Brand']].head())

## 5. Multiple Conditions with `&` (AND) and `|` (OR)

**Critical syntax rule:** Each condition must be wrapped in **parentheses** when combining with `&` or `|`.

| Operator | Meaning | Example |
|----------|---------|---------|
| `&` | AND — both must be True | `(cond1) & (cond2)` |
| `|` | OR — at least one True | `(cond1) | (cond2)` |

> ⚠️ **Common mistake:** Writing `df['A'] > 5 & df['B'] < 10` without parentheses causes a bitwise precedence error. Always wrap each condition!

In [ ]:
# AND: High stock AND high rating
top_products = df[(df['Stock'] > 100) & (df['Rating'] >= 4.5)]
print(f"High stock AND high rating: {len(top_products)} products")
display(top_products[['Name', 'Stock', 'Rating']])

# OR: Expensive OR highly rated
flagship = df[(df['Price'] > 200) | (df['Rating'] >= 4.8)]
print(f"\nExpensive OR highly rated: {len(flagship)} products")
display(flagship[['Name', 'Price', 'Rating']])

## 6. Filtering by Membership with `isin()`

`isin(values)` checks whether each element is contained in `values`. This is much cleaner than chaining multiple `==` conditions with `|`.

**Use case:** Filter rows where a column matches any value in a predefined list.

In [ ]:
# Filter by brand membership
target_brands = ['Logitech', 'Sony', 'Apple']
brand_filter = df[df['Brand'].isin(target_brands)]

print(f"Products from {target_brands}: {len(brand_filter)}")
display(brand_filter[['Name', 'Brand', 'Price']])

# Filter by category membership
audio_vr = df[df['Category'].isin(['Audio', 'VR'])]
print(f"\nAudio or VR products: {len(audio_vr)}")
display(audio_vr[['Name', 'Category', 'Price']])

## 7. Range Filtering with `between()`

`between(left, right, inclusive='both')` selects values within a range. Much more readable than `(x >= left) & (x <= right)`.

| Parameter | Description |
|-----------|-------------|
| `left` | Lower bound |
| `right` | Upper bound |
| `inclusive` | `'both'`, `'left'`, `'right'`, `'neither'` |

In [ ]:
# Products priced between $50 and $150
mid_range = df[df['Price'].between(50, 150)]
print(f"Mid-range products ($50–$150): {len(mid_range)}")
display(mid_range[['Name', 'Brand', 'Price']])

# Stock between 50 and 100 (exclusive of boundaries)
medium_stock = df[df['Stock'].between(50, 100, inclusive='neither')]
print(f"\nMedium stock (50 < stock < 100): {len(medium_stock)}")
display(medium_stock[['Name', 'Stock']])

## 8. String Filtering with `.str` Accessor

The `.str` accessor exposes string methods on Series. These return boolean masks just like numeric comparisons.

| Method | Description |
|--------|-------------|
| `str.contains('text')` | Rows where text appears anywhere |
| `str.startswith('text')` | Rows starting with text |
| `str.endswith('text')` | Rows ending with text |
| `str.match('regex')` | Regex pattern match |
| `str.len()` | String length |

In [ ]:
# Products with 'USB' in the name
usb_products = df[df['Name'].str.contains('USB', case=False, na=False)]
print(f"Products with 'USB' in name: {len(usb_products)}")
display(usb_products[['Name', 'Brand', 'Category']])

# Products starting with 'S'
s_products = df[df['Name'].str.startswith('S', na=False)]
print(f"\nProducts starting with 'S': {len(s_products)}")
display(s_products[['Name', 'Brand']])

# Brands ending with 'r'
r_brands = df[df['Brand'].str.endswith('r', na=False)]
print(f"\nBrands ending with 'r': {len(r_brands)}")
display(r_brands[['Name', 'Brand']])

## 9. SQL-Style Filtering with `query()`

`query(expr)` lets you write filters as strings — great for readability with many conditions. Column names with spaces need backticks.

**Pros:** Clean syntax, easy to parameterize.
**Cons:** Slightly slower than boolean indexing; strings can be harder to debug.

In [ ]:
# Simple query
q1 = df.query("Price > 100")
print(f"Query: Price > 100 → {len(q1)} results")
display(q1[['Name', 'Price']].head())

# Multiple conditions with AND
q2 = df.query("Stock > 50 and Rating >= 4.5")
print(f"\nQuery: Stock > 50 AND Rating >= 4.5 → {len(q2)} results")
display(q2[['Name', 'Stock', 'Rating']])

# Using variables with @
min_price = 50
max_price = 150
q3 = df.query("@min_price <= Price <= @max_price")
print(f"\nQuery: ${min_price} <= Price <= ${max_price} → {len(q3)} results")
display(q3[['Name', 'Price']])

## 10. Negation with `~` (NOT)

The tilde `~` inverts a boolean mask: `True` becomes `False` and vice versa. This is the cleanest way to express "everything except..."

In [ ]:
# Everything EXCEPT black products
non_black = df[~(df['Color'] == 'Black')]
print(f"Non-black products: {len(non_black)}")
display(non_black[['Name', 'Color']])

# Products NOT from Anker or Samsung
exclude_brands = ['Anker', 'Samsung']
other_brands = df[~df['Brand'].isin(exclude_brands)]
print(f"\nProducts NOT from {exclude_brands}: {len(other_brands)}")
display(other_brands[['Name', 'Brand']].head())

## 11. Conditional Replacement with `where()`

`where(cond, other)` keeps original values where the condition is `True`, and replaces them with `other` where `False`. This is the opposite of filtering — instead of dropping rows, you modify them.

**Use case:** Capping values, masking sensitive data, or creating categories.

In [ ]:
# Replace low-stock values with NaN (simulate out-of-stock)
df_where = df.copy()
df_where['Stock'] = df_where['Stock'].where(df_where['Stock'] >= 50, other=0)

print("Stock after where() — items below 50 set to 0:")
display(df_where[['Name', 'Stock']].head(10))

# Create a 'Price Tier' column using where
df_tier = df.copy()
df_tier['Price Tier'] = 'Budget'
df_tier['Price Tier'] = df_tier['Price Tier'].where(df_tier['Price'] < 100, other='Premium')
print("\nPrice tier assignment:")
display(df_tier[['Name', 'Price', 'Price Tier']])

## 12. Filtering by Label Patterns with `filter()`

`filter()` selects rows or columns by their **label names** (not values), using regex or substring matching. It works on the index/column labels, not the cell values.

| Parameter | Description |
|-----------|-------------|
| `items` | Exact list of labels to keep |
| `like` | Substring match |
| `regex` | Regular expression match |
| `axis` | `0` = rows, `1` = columns |

In [ ]:
# Filter columns by name pattern
name_cols = df.filter(like='Name')
print("Columns containing 'Name':")
print(name_cols.columns.tolist())

# Filter columns by regex (columns starting with 'S')
s_cols = df.filter(regex='^S')
print(f"\nColumns starting with 'S': {s_cols.columns.tolist()}")
display(s_cols.head())

# Filter specific columns by exact list
subset = df.filter(items=['Name', 'Brand', 'Price', 'Rating'])
print(f"\nSelected columns: {subset.columns.tolist()}")
display(subset.head())

## 13. Null & Non-Null Filtering

Beyond `isnull().sum()`, you often need to filter rows based on missing data presence. Use `dropna()` or boolean masks with `isnull()` / `notnull()`.

In [ ]:
# Since our dataset has no nulls, let's create some for demonstration
df_demo = df.copy()
df_demo.loc[2, 'Price'] = None
df_demo.loc[5, 'Rating'] = None

# Rows with missing Price
missing_price = df_demo[df_demo['Price'].isnull()]
print("Rows with missing Price:")
display(missing_price[['Name', 'Price']])

# Rows with valid (non-null) Rating
valid_rating = df_demo[df_demo['Rating'].notnull()]
print(f"\nRows with valid Rating: {len(valid_rating)}")

# Drop rows with ANY null value
clean = df_demo.dropna()
print(f"Rows after dropna(): {len(clean)} (removed {len(df_demo) - len(clean)} rows)")

## 14. Chaining Filters Method-Style

Modern pandas encourages method chaining for readable, pipe-like workflows. Use `.loc[]` or `.pipe()` to chain filters without intermediate variables.

In [ ]:
# Method chaining with .loc
result = (df
    .loc[df['Price'] > 50]
    .loc[df['Rating'] >= 4.5]
    .loc[:, ['Name', 'Brand', 'Price', 'Rating', 'Stock']]
    .sort_values('Price', ascending=False)
)

print(f"Chained filter result: {len(result)} products")
display(result)

# Using query in a chain
result2 = (df
    .query("Category in ['Peripherals', 'Audio']")
    .query("Stock > 40")
    .sort_values('Rating', ascending=False)
    [['Name', 'Category', 'Stock', 'Rating']]
)

print(f"\nQuery chain result: {len(result2)} products")
display(result2)

---

## 🎯 Filtering Cheat Sheet

| Goal | Syntax |
|------|--------|
| Equal to | `df[df['Col'] == value]` |
| Not equal | `df[df['Col'] != value]` |
| Greater than | `df[df['Col'] > value]` |
| Range | `df[df['Col'].between(a, b)]` |
| In list | `df[df['Col'].isin([a, b, c])]` |
| String contains | `df[df['Col'].str.contains('text')]` |
| AND (both true) | `df[(cond1) & (cond2)]` |
| OR (either true) | `df[(cond1) | (cond2)]` |
| NOT (invert) | `df[~cond]` |
| SQL-style | `df.query("Col > 10")` |
| Label pattern | `df.filter(like='text')` |
| Conditional replace | `df['Col'].where(cond, other)` |
| Drop nulls | `df.dropna()` |
| Keep non-null | `df[df['Col'].notnull()]` |

### Next Steps
- Combine filtering with `groupby()` for segmented analysis
- Learn `pd.cut()` and `pd.qcut()` for binning continuous data into categories
- Explore `assign()` for adding computed columns during method chains